In [ ]:
# !pip install koreanize_matplotlib

import warnings
import koreanize_matplotlib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as  plt
from matplotlib_venn import venn2
import plotly.express as px
import plotly.graph_objects as go
import ast


# 그래프 해상도 높이기
try:
    %config InlineBackend.figure_format = 'retina'
except Exception as e:
    print(f'💩 {e}')



# 경고 무시
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)  # 출력할 너비를 넉넉하게 조정
pd.set_option('display.expand_frame_repr', False)  # 옆으로 길어져도 줄바꿈 없이 출력
pd.set_option('display.max_colwidth', None)  # 긴 문자열도 생략 없이 출력

try:
    from google.colab import drive
    drive.mount('/content/drive')

    import os
    os.chdir('/content/drive/MyDrive/파트4')
    print('✅ Succesful access google_drive_directory')
    
except Exception as e:
    print('🤗 Hello vscode')
        
## get_df 함수
def get_df(db_name, table_name):
    if table_name in ['accounts_user', 'accounts_blockrecord']:
        table_name = pd.read_parquet(
            f"gs://high_project/{db_name}/{table_name}.parquet", 
            storage_options={'token' : API_KEY_PATH})
    else:
        table_name = pd.read_csv(
            f"gs://high_project/{db_name}/{table_name}.csv",
            storage_options={'token' : API_KEY_PATH}
            )
    return table_name
    

## literal_eval 형변환 함수
def to_literal_eval(df, column):
    return  df[column].apply(lambda x: ast.literal_eval(x) if x != '[]' else [])


# 리스트 내에 드랍 유저가 있는지 확인하는 함수
def find_drop_users(df, column):
    drop_users = [831956, 1580627, 1580689, 1580626, 995177]
    print(f'{column}:')
    for i in drop_users:
        count_drop_rows = len(df[df[column].apply(lambda x: i in x)])
        if count_drop_rows != 0:
            print(f"‼️ 관리자 {i}가 포함된 행 {count_drop_rows}개 존재")
        else:
            print(f"✅ 관리자 {i} 포함행 없음")
            
            
# 데이트타임형으로 변환 및 기간 전처리
def set_datetime(df, column):
    df[column] =  pd.to_datetime(df[column])
    print(f'✅ {column}데이트 타입 형변환 및 기간 전처리 완료')

    return df[df[column] < '2023-09-01']
    
    

            
API_KEY_PATH ='/home/project_yujin/API_KEY/sprintda03-yujin.json'

🤗 Hello vscode


In [126]:
events = get_df('votes', 'events')
events = events[['id', 'title', 'event_type', 'plus_point', 'is_expired', 'created_at']]
set_datetime(events, 'created_at')
events.head()

✅ created_at데이트 타입 형변환 및 기간 전처리 완료


,id,title,event_type,plus_point,is_expired,created_at
0,1,코드잇 은행 가입 이벤트,FCFS,500,1,2023-06-20 11:56:38
1,2,코드잇 멤버십 가입 이벤트,FCFS,1000,1,2023-08-08 07:43:45
2,3,예고 영상 기대평 이벤트,FCFS,500,1,2023-09-24 17:05:59


In [127]:
event_receipts = get_df('votes', 'event_receipts')
event_receipts = event_receipts[['id' ,'user_id' ,'event_id' ,'plus_point' ,'created_at']]
set_datetime(event_receipts, 'created_at')
event_receipts = event_receipts.drop(81)
event_receipts.head()

✅ created_at데이트 타입 형변환 및 기간 전처리 완료


,id,user_id,event_id,plus_point,created_at
0,2,1193618,1,500,2023-06-22 09:25:16
1,3,928351,1,500,2023-06-22 09:38:53
2,4,904872,1,500,2023-06-22 10:32:15
3,5,974697,1,500,2023-06-22 13:03:06
4,6,1168260,1,500,2023-06-22 13:40:38


In [128]:
accounts_paymenthistory = get_df('votes', 'accounts_paymenthistory')
set_datetime(accounts_paymenthistory, 'created_at')
accounts_paymenthistory.head()

✅ created_at데이트 타입 형변환 및 기간 전처리 완료


,id,productId,phone_type,created_at,user_id
0,6,heart.777,A,2023-05-13 21:28:34,1211127
1,7,heart.777,A,2023-05-13 21:29:39,1151343
2,8,heart.777,A,2023-05-13 21:31:33,1002147
3,9,heart.777,A,2023-05-13 21:31:39,1095040
4,11,heart.777,A,2023-05-13 21:34:32,1164081


In [129]:
accounts_failpaymenthistory = get_df('votes', 'accounts_failpaymenthistory')
set_datetime(accounts_failpaymenthistory, 'created_at')
accounts_failpaymenthistory.head()

✅ created_at데이트 타입 형변환 및 기간 전처리 완료


,id,productId,phone_type,created_at,user_id
0,6,heart.200,A,2023-05-14 05:49:22,1055891
1,7,heart.777,A,2023-05-14 08:17:21,1152151
2,8,heart.777,A,2023-05-14 10:11:46,986200
3,9,heart.1000,A,2023-05-14 11:53:09,1028261
4,10,heart.777,A,2023-05-14 12:30:47,1235730


In [136]:
'''데이터 불러오기 및 컬럼순서 별경'''
accounts_pointhistory = get_df('votes', 'accounts_pointhistory')
accounts_pointhistory = accounts_pointhistory[['id', 'user_id', 'user_question_record_id', 'delta_point', 'created_at']]

'''중복값 재거'''
accounts_pointhistory = accounts_pointhistory[~accounts_pointhistory.loc[:, [ 'user_id', 'user_question_record_id', 'delta_point', 'created_at']].duplicated()]
# accounts_pointhistory = accounts_pointhistory.dropna()

'''8월까지 날짜 필터링'''
# accounts_pointhistory['user_question_record_id'] = accounts_pointhistory['user_question_record_id'].astype(int)
set_datetime(accounts_pointhistory, 'created_at')
accounts_pointhistory.head()

✅ created_at데이트 타입 형변환 및 기간 전처리 완료


,id,user_id,user_question_record_id,delta_point,created_at
0,790629,849436,771777.0,9,2023-04-28 12:27:49
1,790652,849436,771800.0,9,2023-04-28 12:28:02
2,790664,849436,771812.0,5,2023-04-28 12:28:09
3,790680,849436,771828.0,13,2023-04-28 12:28:16
4,790703,849436,771851.0,5,2023-04-28 12:28:26


In [147]:
print(sorted(accounts_pointhistory[accounts_pointhistory['delta_point'] > 0]['delta_point'].unique().tolist()))

[5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 200, 210, 220, 230, 240, 250, 260, 270, 280, 300, 500, 777, 1000]


In [149]:
print(sorted(accounts_pointhistory[accounts_pointhistory['delta_point'] < 0]['delta_point'].unique().tolist()))

[-1000, -500, -300, -200, -30, -10]


In [108]:
print(event_receipts['plus_point'].unique())
praccounts_paymenthistory['productId'].unique()

[ 500 1000]
['heart.777' 'heart.200' 'heart.4000' 'heart.1000']


In [131]:
# 포인트 관련 모든 기록에서 포인트 델타가 200, 777, 1000, 4000 플러스인것들
# 포인트 구매기록과의 비교
filter_point_hist = accounts_pointhistory[accounts_pointhistory['delta_point'].isin([200, 777, 1000, 4000])]
accounts_paymenthistory 

print(f"filter_point_hist: {filter_point_hist['created_at'].min()} ~ {filter_point_hist['created_at'].max()}")
print(f"payment_history: {accounts_paymenthistory['created_at'].min()} ~ {accounts_paymenthistory['created_at'].max()}")

filter_point_hist: 2023-06-22 09:01:32 ~ 2024-02-27 14:41:47
payment_history: 2023-05-13 21:28:34 ~ 2024-05-08 14:12:45


In [133]:
accounts_paymenthistory.query('user_id == 1086654')

,id,productId,phone_type,created_at,user_id
95134,98073,heart.777,I,2024-05-06 14:51:27,1086654
95135,98074,heart.777,I,2024-05-06 14:51:27,1086654


In [135]:
accounts_pointhistory.query('user_id == 1086654')

,id,user_id,user_question_record_id,delta_point,created_at


In [112]:
filter_point_hist = filter_point_hist[filter_point_hist['created_at'] >= '2023-06-22 09:01:32']
filter_accounts_paymenthistory = accounts_paymenthistory[accounts_paymenthistory['created_at'] >= '2023-06-22 09:01:32']

filter_point_hist = filter_point_hist[filter_point_hist['created_at'] <= '2024-02-27 14:41:47']
filter_accounts_paymenthistory = accounts_paymenthistory[accounts_paymenthistory['created_at'] <= '2024-02-27 14:41:47']

In [115]:
df = pd.concat([filter_point_hist, filter_accounts_paymenthistory]).sort_values(by=['user_id', 'created_at'])

In [123]:
df['user_id'].value_counts(0)

user_id
1527451    60
1246471    51
1141603    35
1204373    34
1099530    27
           ..
1194461     1
1194467     1
1194493     1
1194522     1
1582793     1
Name: count, Length: 59834, dtype: int64

In [150]:
accounts_userquestionrecord = get_df('votes', 'accounts_userquestionrecord')

# 컬럼 순서 정리
accounts_userquestionrecord = accounts_userquestionrecord[['id', 'user_id', 'chosen_user_id', 'question_id', 'question_piece_id', \
    'status' ,'answer_status', 'answer_updated_at', 'has_read', 'opened_times', 'report_count', 'created_at']]

# 상태컬럼, 응답상태컬럼 한글로 매핑
accounts_userquestionrecord['status'] = accounts_userquestionrecord['status'].replace({'C':'닫힘'}).replace({'I':'초성열림'}).replace({'B':'차단'})
accounts_userquestionrecord['answer_status'] = accounts_userquestionrecord['answer_status'].replace({'N':'미답변'}).replace({'P':'비공개'}).replace({'A':'공개'})

# 시간타입 변경 및 8월까지 필터링
set_datetime(accounts_userquestionrecord, 'answer_updated_at')
set_datetime(accounts_userquestionrecord, 'created_at')

# accounts_userquestionrecord['diff_time'] = accounts_userquestionrecord['answer_updated_at'] - accounts_userquestionrecord['created_at']
# accounts_userquestionrecord['diff_sec_time'] = (accounts_userquestionrecord['answer_updated_at'] - accounts_userquestionrecord['created_at']).dt.total_seconds()

✅ answer_updated_at데이트 타입 형변환 및 기간 전처리 완료
✅ created_at데이트 타입 형변환 및 기간 전처리 완료


In [152]:
accounts_pointhistory.head()

,id,user_id,user_question_record_id,delta_point,created_at
0,790629,849436,771777.0,9,2023-04-28 12:27:49
1,790652,849436,771800.0,9,2023-04-28 12:28:02
2,790664,849436,771812.0,5,2023-04-28 12:28:09
3,790680,849436,771828.0,13,2023-04-28 12:28:16
4,790703,849436,771851.0,5,2023-04-28 12:28:26


In [153]:
accounts_userquestionrecord.head()

,id,user_id,chosen_user_id,question_id,question_piece_id,status,answer_status,answer_updated_at,has_read,opened_times,report_count,created_at
0,771777,849436,849469,252,998458,닫힘,미답변,2023-04-28 12:27:49,0,0,0,2023-04-28 12:27:49
1,771800,849436,849446,244,998459,닫힘,미답변,2023-04-28 12:28:02,0,0,0,2023-04-28 12:28:02
2,771812,849436,849454,183,998460,닫힘,미답변,2023-04-28 12:28:09,1,0,0,2023-04-28 12:28:09
3,771828,849436,847375,101,998461,닫힘,미답변,2023-04-28 12:28:16,0,0,0,2023-04-28 12:28:16
4,771851,849436,849477,209,998462,닫힘,미답변,2023-04-28 12:28:26,1,0,0,2023-04-28 12:28:26


In [168]:
def get_date_range(df1, col1, df1_name, df2, col2, df2_name):
    print(f"{df1_name}: {df1[col1].min()} ~ {df1[col1].max()}")
    print(f"{df2_name}: {df2[col2].min()} ~ {df2[col2].max()}")
    
    start_date = max(df1[col1].min(), df2[col2].min())
    end_date = min(df1[col1].max(), df2[col2].max())
    
    print(f"두 시간 범위의 교집합: {start_date} ~ {end_date}")

In [170]:
get_date_range(accounts_userquestionrecord, 'created_at', '질문기록', accounts_pointhistory, 'created_at', '포인트 증감 기록')

질문기록: 2023-04-28 12:27:49 ~ 2024-05-08 01:36:18
포인트 증감 기록: 2023-04-28 12:27:49 ~ 2024-05-08 01:36:18
두 시간 범위의 교집합: 2023-04-28 12:27:49 ~ 2024-05-08 01:36:18
